In [ ]:
import ast
import numpy as np

#local, global in cluster setting

# Define file paths
base_path = "/home/kamrul/Documents/kamrul_files_Linux/OORT/ClusterFed_OORT/results/svhn2/"

# files = {
#     'Proposed': base_path + 'output_svhn_custom_local_dirichlet_Q8_lr0.01_ep1.txt',
#     'OORT': base_path + 'output_svhn_oort_local_dirichlet.txt',
#     'Random': base_path + 'output_svhn_random_local_dirichlet_Q8_lr0.01_ep1.txt',
# }

In [ ]:
#global in non-cluster setting

# files = {
#     'Proposed': base_path + 'output_mnist_custom_global_dirichlet_Q8_lr0.01_ep1_w0.4.txt',
#     'OORT': base_path + 'output_mnist_oort_global_dirichlet1.txt',
# }

# files = {
#     'Proposed': base_path + 'output_fmnist_custom_global_dirichlet_Q8_lr0.01_ep1_w0.4.txt',
#     'OORT': base_path + 'output_fmnist_oort_global_dirichlet1.txt',
# }

# files = {
#     'Proposed': base_path + 'output_cifar10_custom_global_dirichlet_Q8_lr0.01_ep1_w0.4.txt',
#     'OORT': base_path + 'output_cifar10_oort_global_dirichlet1.txt',
# }

# files = {
#     'Proposed': base_path + 'output_svhn_custom_global_dirichlet_Q8_lr0.01_ep1_w0.4.txt',
#     'OORT': base_path + 'output_svhn_oort_global_dirichlet1.txt',
# }


In [ ]:
# Define a function to extract list from text file
def extract_accuracy_list(file_path, list_name):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    for line in lines:
        if list_name in line:
            return ast.literal_eval(line.split('=')[1].strip())
    raise ValueError(f"{list_name} not found in {file_path}")

# Define a function to get first round that reaches a target accuracy
def round_to_reach_threshold(accuracy_list, threshold):
    for i, acc in enumerate(accuracy_list):
        if acc >= threshold:
            return i + 1  # +1 since rounds are 1-indexed
    return float('nan')

# Define thresholds and structure
thresholds = [60, 65, 70, 75, 80, 85, 90]
#thresholds = [40, 45, 50, 55, 60]
result_table = {}

# Process each method
for method, path in files.items():
    list_name = {
        'Proposed': 'accuracy_custom',
        'OORT': 'accuracy_oort',
        'Random': 'accuracy_random',
    }[method]
    
    accuracy = extract_accuracy_list(path, list_name)
    
    result_table[method] = {
        f'@{thr}': round_to_reach_threshold(accuracy, thr) for thr in thresholds
    }
    result_table[method]['Best Accuracy (%)'] = max(accuracy)

# Print result in a readable format (or convert to LaTeX/DataFrame if needed)
print(f"{'Method':<10} {'@60':>10} {'@65':>10} {'@70':>10} {'@75':>10} {'@80':>10} {'@85':>10} {'@90':>10} {'Best Accuracy (%)':>20}")
#print(f"{'Method':<10} {'@40':>10} {'@45':>10} {'@50':>10} {'@55':>10} {'@60':>10} {'Best Accuracy (%)':>20}")
for method, values in result_table.items():
    print(f"{method:<10} {values['@60']:>10} {values['@65']:>10} {values['@70']:>10} {values['@75']:>10} {values['@80']:>10} {values['@85']:>10}  {values['@90']:>10} {values['Best Accuracy (%)']:>20.2f}")
    #print(f"{method:<10} {values['@40']:>10} {values['@45']:>10} {values['@50']:>10} {values['@55']:>10} {values['@60']:>10} {values['Best Accuracy (%)']:>20.2f}")

In [ ]:
def compute_improvement_ratios(result_table, reference='Proposed'):
    thresholds = [key for key in result_table[reference] if key.startswith('@')]
    baselines = [k for k in result_table if k != reference]

    improvements = {}

    for baseline in baselines:
        time_improvements = []
        for thr in thresholds:
            proposed_round = result_table[reference][thr]
            baseline_round = result_table[baseline][thr]
            if not np.isnan(proposed_round) and not np.isnan(baseline_round) and proposed_round > 0:
                ratio = baseline_round / proposed_round
                time_improvements.append(ratio)
        
        # Final accuracy improvement
        acc_prop = result_table[reference]['Best Accuracy (%)']
        acc_base = result_table[baseline]['Best Accuracy (%)']
        acc_improvement = acc_prop / acc_base if acc_base > 0 else float('nan')

        improvements[baseline] = {
            'avg_time_to_accuracy_improvement': np.mean(time_improvements) if time_improvements else float('nan'),
            'final_accuracy_improvement': acc_improvement
        }

    return improvements

# Example: compute and print
improvements = compute_improvement_ratios(result_table)

for method, vals in improvements.items():
    time_ratio = vals['avg_time_to_accuracy_improvement']
    acc_ratio = vals['final_accuracy_improvement']
    print(f"\nCompared to {method}:")
    if not np.isnan(time_ratio):
        print(f"- Time-to-Accuracy Improvement: {time_ratio:.2f}X")
    else:
        print(f"- Time-to-Accuracy Improvement: N/A")
    
    if not np.isnan(acc_ratio):
        print(f"- Final Accuracy Improvement: {acc_ratio:.2f}X")
    else:
        print(f"- Final Accuracy Improvement: N/A")
